# 🧾 1.1 Conversión FASTA → CSV

Este notebook constituye el primer paso del pipeline de datos: convierte las secuencias de péptidos en formato **FASTA** a una tabla estructurada **CSV**, dejándolas listas para las etapas posteriores de limpieza, etiquetado y generación de embeddings.

---

## 📋 Descripción general

El notebook opera sobre archivos FASTA crudos y produce una representación tabular donde cada fila corresponde a una secuencia. En general:

- **Entrada:** uno o varios archivos `.fasta` con secuencias de péptidos (positivos/negativos).
- **Salida:** un archivo `.csv` con, al menos, la secuencia y su etiqueta/clase.

Flujo típico:
1. Lectura de los registros FASTA (identificador + secuencia).
2. Normalización básica (mayúsculas, eliminación de caracteres inválidos).
3. Asignación de etiquetas según el origen del archivo (AMP / no-AMP).
4. Escritura del resultado en un CSV consolidado.

---

## 🛠️ Funcionalidades principales

1. **Parsing de FASTA**
   - Lectura secuencial de identificadores y secuencias.
   - Manejo de entradas multilínea y formatos comunes.

2. **Limpieza y estandarización**
   - Normalización de caracteres y longitud.
   - Detección y filtrado de secuencias vacías o corruptas.

3. **Etiquetado**
   - Asignación de la clase binaria a cada secuencia.
   - Consolidación de múltiples fuentes en una sola tabla.

4. **Exportación**
   - Escritura de un CSV listo para el preprocesamiento posterior.
   - Registro de conteos por clase para verificar el balance inicial.

## 📦 Instalación de dependencias

In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.1 MB/s eta 0:00:00


## 🔧 Importación de librerías

In [2]:
import os
import glob
import pandas as pd
from Bio import SeqIO
from pathlib import Path

## ⚙️ Configuración de parámetros

In [ ]:
INPUT_FASTA = "/kaggle/input/datasets/user/fasta-input"
OUTPUT_CSV = "/kaggle/working/"
os.makedirs(OUTPUT_CSV, exist_ok=True)

MIN_LEN = 10
MAX_LEN = 100
PAD_CHAR = 'X'

# 🆕 Conjunto de aminoácidos válidos (20 estándar + 'X' para padding y caracteres desconocidos/no estándar)
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYX")

In [4]:
def clean_sequence(seq: str) -> str:
    """Reemplaza caracteres no estándar por 'X'."""
    return "".join(aa if aa in VALID_AA else 'X' for aa in seq.upper())

## 🔄 Función de conversión FASTA → CSV

In [5]:
def fasta_to_csv(
    input_fasta: str, 
    output_csv: str, 
    min_len: int = MIN_LEN, 
    max_len: int = MAX_LEN, 
    pad_char: str = PAD_CHAR
) -> pd.DataFrame:
    """
    Convierte un archivo FASTA a CSV con validación, recorte, relleno y estadísticas.
    """
    try:
        records = []
        for record in SeqIO.parse(input_fasta, "fasta"):
            raw_seq = str(record.seq).upper()
            records.append({
                "id": record.id,
                "description": record.description,
                "sequence": raw_seq,
                "original_length": len(raw_seq) # Preservar longitud original
            })

        df = pd.DataFrame(records)
        if df.empty:
            print(f"⚠️ {Path(input_fasta).name} is empty. Omitted.")
            return df

        df_processed = df.copy()
        
        # 1. Validar y limpiar caracteres
        df_processed["sequence"] = df_processed["sequence"].apply(clean_sequence)

        # 2. Trim sequences > MAX_LEN
        mask_long = df_processed["original_length"] > max_len
        if mask_long.any():
            df_processed.loc[mask_long, "sequence"] = df_processed.loc[mask_long, "sequence"].str[:max_len]

        # 3. Pad sequences < MIN_LEN
        mask_short = df_processed["original_length"] < min_len
        if mask_short.any():
            df_processed.loc[mask_short, "sequence"] = df_processed.loc[mask_short, "sequence"].str.ljust(min_len, pad_char)

        # 4. Recalculate final length
        df_processed["length"] = df_processed["sequence"].str.len()

        # 5. Detectar duplicados
        dupes = df_processed["sequence"].duplicated().sum()

        # 6. Estadísticas de distribución
        orig_min, orig_max = df_processed['original_length'].min(), df_processed['original_length'].max()
        
        print(f"✅ {Path(input_fasta).name} → {Path(output_csv).name}")
        print(f"   📊 Total: {len(df_processed)} | ✂️ Trimmed: {mask_long.sum()} | 📝 Padded: {mask_short.sum()} | ⚠️ Dupes: {dupes}")
        print(f"   📏 Original Length -> min: {orig_min}, max: {orig_max}")

        # Guardar
        df_processed.to_csv(output_csv, index=False)
        return df_processed

    except Exception as e:
        print(f"❌ Error crítico procesando {Path(input_fasta).name}: {e}")
        return pd.DataFrame()

## 📂 Procesamiento por lotes

In [6]:
fasta_files = glob.glob(os.path.join(INPUT_FASTA, "*.fasta"))

if not fasta_files:
    print(f"⚠️ No .fasta files were found in: {INPUT_FASTA}")
else:
    print(f"📂 Processing {len(fasta_files)} FASTA file(s)...\n")
    
    manifest_data = []
    
    for fasta_path in sorted(fasta_files):
        nombre = Path(fasta_path).stem
        csv_path = os.path.join(OUTPUT_CSV, f"{nombre}.csv")
        
        df_result = fasta_to_csv(fasta_path, csv_path)
        
        # Guardar datos para el manifiesto global
        if not df_result.empty:
            manifest_data.append({
                "file": nombre,
                "total_sequences": len(df_result),
                "original_min_len": df_result["original_length"].min(),
                "original_max_len": df_result["original_length"].max(),
                "duplicates": df_result["sequence"].duplicated().sum()
            })
            
        print("-" * 60)

    # Exportar manifiesto global
    if manifest_data:
        df_manifest = pd.DataFrame(manifest_data)
        manifest_path = os.path.join(OUTPUT_CSV, "_processing_manifest.csv")
        df_manifest.to_csv(manifest_path, index=False)

📂 Processing 6 FASTA file(s)...

✅ test_neg.fasta → test_neg.csv
   📊 Total: 10771 | ✂️ Trimmed: 0 | 📝 Padded: 0 | ⚠️ Dupes: 0
   📏 Original Length -> min: 11, max: 100
------------------------------------------------------------
✅ test_pos.fasta → test_pos.csv
   📊 Total: 4914 | ✂️ Trimmed: 0 | 📝 Padded: 267 | ⚠️ Dupes: 0
   📏 Original Length -> min: 5, max: 100
------------------------------------------------------------
✅ train_neg.fasta → train_neg.csv
   📊 Total: 9767 | ✂️ Trimmed: 0 | 📝 Padded: 926 | ⚠️ Dupes: 0
   📏 Original Length -> min: 5, max: 100
------------------------------------------------------------
✅ train_pos.fasta → train_pos.csv
   📊 Total: 9781 | ✂️ Trimmed: 0 | 📝 Padded: 1054 | ⚠️ Dupes: 0
   📏 Original Length -> min: 5, max: 100
------------------------------------------------------------
✅ val_neg.fasta → val_neg.csv
   📊 Total: 2561 | ✂️ Trimmed: 0 | 📝 Padded: 244 | ⚠️ Dupes: 0
   📏 Original Length -> min: 5, max: 100
----------------------------------------

## 🏁 Resultado final: dataset tabular consolidado

El notebook cumple su objetivo de transformar las secuencias crudas en formato FASTA a una tabla CSV estructurada y etiquetada, que sirve como entrada para la generación de embeddings y el entrenamiento de modelos.

### Entregables

| Entregable | Formato | Descripción |
|---|---|---|
| Dataset consolidado | `.csv` | Secuencias + etiqueta binaria |
| Conteos por clase | salida estándar | Balance inicial de positivos/negativos |

### Análisis

- **Trazabilidad:** cada secuencia conserva su identificador de origen, lo que permite rastrear su procedencia a lo largo del pipeline.
- **Calidad de datos:** la normalización y el filtrado reducen ruido antes de la etapa de embeddings.
- **Interoperabilidad:** el formato CSV es compatible con el resto de notebooks de preprocesamiento y modelado.

**Estado final:** el CSV generado queda disponible para la etapa de **preprocesamiento de entradas** y posterior cálculo de embeddings ProtFlash.